In [13]:
from typing_extensions import TypedDict
from typing import List
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from pydantic import BaseModel

llm = init_chat_model("openai:gpt-4o")

In [14]:
class State(TypedDict):
    dish: str
    ingredients: list[dict]
    recipe_steps: str
    plating_instructions: str

class Ingredient(BaseModel):
    name: str
    quantity: str
    unit: str

class IngredientsOutput(BaseModel):
    ingredients: List[Ingredient]

In [15]:
def list_ingredients(state: State):
    structured_llm = llm.with_structured_output(IngredientsOutput)
    response = structured_llm.invoke(f"{state['dish']}을 만드는 데 필요한 재료 5~15가지만 알려줘.")  
    return {"ingredients": response.ingredients}

def create_recipe(state: State):
    response = llm.invoke(f"{state['ingredients']}을(를) 사용하여 {state['dish']}을(를) 만드는 단계별 요리법을 작성해줘")
    return {
        "recipe_steps": response.content
    }

def describe_plating(state: State):
    response = llm.invoke(f"이 레시피 {state["recipe_steps"]}를 바탕으로 {state["dish"]}를 예쁘게 플레이팅하는 방법을 설명해")
    return {
        "plating_instructions": response.content
    }

In [16]:
graph_builder = StateGraph(State)

graph_builder.add_node("list_ingredients", list_ingredients)
graph_builder.add_node("create_recipe", create_recipe)
graph_builder.add_node("describe_plating", describe_plating)

graph_builder.add_edge(START, "list_ingredients")
graph_builder.add_edge("list_ingredients", "create_recipe")
graph_builder.add_edge("create_recipe", "describe_plating")
graph_builder.add_edge("describe_plating", END)

graph = graph_builder.compile()

In [ ]:
graph.invoke({"dish": "두바이쫀득쿠키"})

{'dish': '김치찌개',
 'ingredients': [Ingredient(name='배추 김치', quantity='200', unit='g'),
  Ingredient(name='돼지고기', quantity='100', unit='g'),
  Ingredient(name='두부', quantity='150', unit='g'),
  Ingredient(name='양파', quantity='1', unit='개'),
  Ingredient(name='대파', quantity='1', unit='줄기'),
  Ingredient(name='새우젓', quantity='1', unit='큰술'),
  Ingredient(name='다시마', quantity='1', unit='조각'),
  Ingredient(name='고춧가루', quantity='1.5', unit='큰술'),
  Ingredient(name='간장', quantity='1', unit='큰술'),
  Ingredient(name='마늘', quantity='2', unit='쪽'),
  Ingredient(name='소금', quantity='약간', unit=''),
  Ingredient(name='후추', quantity='약간', unit=''),
  Ingredient(name='물', quantity='500', unit='ml')],
 'recipe_steps': '김치찌개는 한국의 대표적인 찌개 요리 중 하나로, 발효된 배추 김치와 다양한 재료를 함께 끓여 풍부한 맛을 내는 것이 특징입니다. 주어진 재료를 사용하여 김치찌개를 만드는 방법은 다음과 같습니다:\n\n### 재료:\n- 배추 김치: 200g\n- 돼지고기: 100g\n- 두부: 150g\n- 양파: 1개\n- 대파: 1줄기\n- 새우젓: 1 큰술\n- 다시마: 1조각\n- 고춧가루: 1.5 큰술\n- 간장: 1 큰술\n- 마늘: 2쪽\n- 소금 약간\n- 후추 약간\n- 물: 500ml\n\n### 만들기:\